# Page to Pixel—Digitization, OCR, and Making Texts into Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/page-to-pixel/blob/main/Page-to-Pixel.ipynb)

In this notebook, we'll explore some of the different ways that the characters and words in digitized images of texts can be converted into digital formats. We'll work with one of the most commonly used Optical Character Recognition (OCR) engines, Tesseract, in its Python wrapper, `pytesseract`. We'll also test out some of the most recent Vision Language Models (VLMs), which adapt the technology of Large Language Models (LLMs) to interpret and then generate text. Feel free to upload your own images and test out the ability of the different models to correctly extract the text. 

In [ ]:
""" If your're working in Colab, you can execute this cell to bring everything, 
including the sample data, over from the GitHub repository so that you can have 
it all in one place"""
!git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/page-to-pixel.git
%cd page-to-pixel

Cloning into 'page-to-pixel'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 18 (delta 0), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 23.97 MiB | 31.27 MiB/s, done.
/Users/bradykrien/Workspace/text-to-data_workshops/page-to-pixel/page-to-pixel


In [ ]:
# Packages to install if necessary
!pip install pytesseract # main OCR package
!pip install Pillow #Python Image Library is the main image package
!pip install pandas # needed for pytesseract's DATAFRAME output
!pip install -q -U transformers accelerate bitsandbytes torchvision # ML and VLM facilitation packages (-U for newer architectures like PaddleOCR-VL and Qwen3.5; torchvision is needed by the image/video processors)


In [ ]:
# Packages to import
import pytesseract # Classic OCR package in Python
import torch # PyTorch is a deep learning framework
from PIL import Image # Python image processing library
from pytesseract import Output # import Output module for reporting data
import pandas as pd # pandas is useful for representing and manipulating data
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

## Tesseract OCR Library

### Dickens

We have a page from Charles Dickens' *A Tale of Two Cities* from the HathiTrust Digital Library. This image has a single column of printed text, which makes it a good candidate for basic OCR processing. The original image can be access here: [https://babel.hathitrust.org/cgi/pt?id=hvd.32044024216053&seq=21](https://babel.hathitrust.org/cgi/pt?id=hvd.32044024216053&seq=21)

In [ ]:
# Let's see what the text it outputs looks like
dickens = Image.open('sample_data/dickens.jpg') # Open the image
print(pytesseract.image_to_string(dickens)) # Process the image and output the text

A TALE OF TWO CITIES.
IN THREE BOOKS.

BY

CHARLES DICKENS.
VOL. L

ET

BOOK THE FIRST.

RECALLED TO LIFE.

CHAPTER I.
The Period.

Iv was the best of times, it was the worst of times,
it was the age of wisdom, it was the age of foolishness,
it was the epoch of belief, it was the epoch of incre-
dulity, it was the season of Light, it was the season
of Darkness, it was the spring of hope, it was the win-
ter of despair, we had everything before us, we had
nothing before us, we were all going direct to Heaven,
we were all going direct the other way — in short, the
period was so far like the present period, that some of
its noisiest authorities insisted on its being received,
_for good or for evil, in the superlative degree of com-
parison only. * 1%

Google



### Getting confidence scores

`image_to_string` only returns plain text. To get Tesseract's per-word confidence (0-100, or -1 for non-text regions like lines), use `image_to_data`, which also gives bounding boxes for each recognized word.

In [7]:
dickens_data = pytesseract.image_to_data(dickens, output_type=Output.DATAFRAME) # Let's take a look at the data 

# Drop rows with no recognized text (conf == -1) and preview word-level confidence
words = dickens_data[dickens_data.conf != -1][['text', 'conf', 'left', 'top', 'width', 'height']]
words.head(20)

,text,conf,left,top,width,height
4,A,96.774300,477,729,65,69
5,TALE,95.865318,598,726,260,74
6,OF,96.161652,917,723,129,77
7,TWO,95.899612,1101,724,229,77
8,CITIES.,96.249382,1386,722,332,78
10,IN,96.246429,815,877,65,42
11,THREE,96.422119,917,876,204,42
12,BOOKS.,96.494560,1154,875,213,43
16,BY,96.655952,1058,1013,72,36
20,CHARLES,95.875282,722,1095,354,57


In [9]:
# A single overall confidence score for the page
print(f"Average word confidence for the dickens text: {words.conf.mean():.1f}")

Average word confidence for the dickens text: 93.5


### The Gettysburg Address

Our next image is significantly more complicated. It's a handwritten draft of Lincoln's "Gettysburg Address." Handwriting, even relatively neat and legible handwriting like Lincoln's, is much less consistent that printed text. This image is from the Library of Congress and can be accessed at: [https://www.loc.gov/resource/mal.4356600/?st=text&r=-0.066,-0.288,1.111,1.432,0](https://www.loc.gov/resource/mal.4356600/?st=text&r=-0.066,-0.288,1.111,1.432,0)

In [10]:
# Again, we'll start with the text output
gettysburg = Image.open('sample_data/gettysburg.jpg') # Open the image
print(pytesseract.image_to_string(gettysburg)) # Process the image and output the text

ruv ecme anev seven yeors age Cry forbes
bange frk, pow KA Coutenend, a henwr ration, Cont
Cewees tw VberS” an clecteelcn 6 Be frgftentins
VOTE OLE! Piven, Gags Cie CeO agrck,

Now pres Be Prgepeed pe yore Cink way Ge:
“ey whether LKaD ator oy Ang Malach, bo Conceunal,
Prov fo clectes€ev Con beg preter, Ne aw Mer
PS re a ped batho feted fe ie

meds: eee Clecthitvets a polos Ge ad ite fica

Prev fvcfier AAEO bres pho Cs ale ee,

Pe ew ae Farge br 20 We Caw [rut clectesta—
We Cen (hoC Coiecie Wwe Caw hod halen ges
Y acenanie tke Qs Trew fatting
poor
, mia dics as Cokzgen geo tO fe aleve cu, ower
be acto or eCOrzee, DE wWorkev nr £6 Le, i
(ov fog pemmeintey t\hal Gr Hay. Keres Via i

vey




In [12]:
# Let's take a look at the data and, in particular, the confidence:
gettysburg_data = pytesseract.image_to_data(gettysburg, output_type=Output.DATAFRAME) # Let's take a look at the data 

# Drop rows with no recognized text (conf == -1) and preview word-level confidence
words = gettysburg_data[gettysburg_data.conf != -1][['text', 'conf', 'left', 'top', 'width', 'height']]
words.head(20)

,text,conf,left,top,width,height
4,ruv,0.000000,1461,249,94,52
5,ecme,0.000000,1583,245,105,84
6,anev,58.477577,1716,245,91,84
7,seven,0.484810,1835,245,119,84
8,yeors,35.119865,1982,245,126,84
9,age,69.577896,2125,266,83,62
10,Cry,61.301174,2225,256,65,65
11,forbes,43.025341,2311,256,126,65
13,bange,0.000000,1420,339,143,70
14,"frk,",67.052475,1572,349,114,55


In [ ]:
# A single overall confidence score for the page
print(f"Average word confidence for the gettysburg text: {words.conf.mean():.1f}")

Average word confidence for the dickens text: 33.5


## Vision Language Models

While Vision Language Models (VLMs) perform a similar task to traditional Optical Character Recognition tools like Tesseract, they go about it in a very different way. They are partly interpretive but also partly generative, which means that, when applied to a text they can't correctly process, they are much less likely to output meaningless strings of characters but much more likely to output coherent but incorrect information. 

In [ ]:
# Both VLMs below will use whatever accelerator is available
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

### General Vision Language Model 
Use the Qwen model here

[Qwen3.5](https://huggingface.co/Qwen/Qwen3.5-0.8B) is Alibaba's newest Qwen release. Unlike the earlier Qwen2.5-VL, which bolted a vision encoder onto a text-only model, Qwen3.5 is trained natively on interleaved text, image, and video tokens from the start. It's a general-purpose chat model, not an OCR specialist, so we prompt it conversationally and give it our handwritten Gettysburg Address image.

In [ ]:
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-0.8B")
qwen_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    dtype="bfloat16" if device != "cpu" else "float32",
).to(device)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": gettysburg},
            {"type": "text", "text": "Transcribe all of the text in this image."},
        ],
    }
]

qwen_inputs = qwen_processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(qwen_model.device)

qwen_outputs = qwen_model.generate(**qwen_inputs, max_new_tokens=1024)
qwen_result = qwen_processor.decode(
    qwen_outputs[0][qwen_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(qwen_result)

### OCR Vision Language Model
Use the GLM-OCR model here.

[GLM-OCR](https://huggingface.co/zai-org/GLM-OCR) is a compact (0.9B parameter) document-parsing VLM from Zhipu AI (Z.ai), currently ranked #1 on the OmniDocBench V1.5 leaderboard. Like PaddleOCR-VL, it's a specialist rather than a general chat model, but it's driven the same way we prompted Qwen3.5 above — through a chat template — with a short task instruction such as `"OCR:"`.

In [12]:
glm_processor = AutoProcessor.from_pretrained("zai-org/GLM-OCR")
glm_model = AutoModelForImageTextToText.from_pretrained(
    "zai-org/GLM-OCR",
    dtype="bfloat16" if device != "cpu" else "float32",
).to(device)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": gettysburg},
            {"type": "text", "text": "OCR:"},
        ],
    }
]

glm_inputs = glm_processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(glm_model.device)
glm_inputs.pop("token_type_ids", None)  # GLM-OCR's processor adds this, but generate() doesn't accept it

glm_outputs = glm_model.generate(**glm_inputs, max_new_tokens=1024)
glm_result = glm_processor.decode(
    glm_outputs[0][glm_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(glm_result)

Loading weights: 100%|██████████| 510/510 [00:00<00:00, 18667.05it/s]


Four score and seven years ago our fathers brought forth, upon this continent, a new nation, cow:
ceewed in liberty, and dedicated to the proportion
that all men are created equal.

Now we are engaged in a great civil war, test:
ing whether that nation, or any nation, so conceived,
and so dedicated, can long endure. We are met
here on a great battle-field of their war. We have
come to deduce a portion of it as the final part:
ing places of those who here gave their lives, that
that nation might live. It is altogether fitting
and proper that we should do this.

But in a larger sense we can not deduce
we can not consecrate we can not hallow this
ground. The brave men, living and dead, who slung
glue here, have consecrated it far above our power
to add or detract. The world will little note,
nor long remember, what we say here, but
can never forget what they did here. It is
for us, the living, rather to be dedicated
how to this unfinished, which they have,
thus far, so nobly carried on. I